# Quantitative EEG Analysis Across Sleep Stages

A spectral analysis of EEG activity during different sleep stages using real Sleep-EDF data.

**Recording:** SC4001 (Sleep-EDF Expanded)  
**Channel:** EEG Fpz-Cz (100 Hz sampling rate)  
**Epochs:** 30-second non-overlapping windows  
**Stages:** Wake, N1, N2, N3, REM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import sys
import os

# Use the project's data loader
sys.path.insert(0, '.')
from data_loader import load_data, STAGE_NAMES

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## 1. Load Data

Load the PSG recording and aligned hypnogram labels.

In [ ]:
X, y, sfreq, epoch_samples = load_data()
print(f"\nDataset shape: {X.shape} (epochs x samples)")
print(f"Sampling rate: {sfreq} Hz")

---
## 2. Power Spectral Density per Stage

We use Welch's method to estimate the PSD for every epoch, then average within each sleep stage. Welch's method reduces noise by dividing the signal into overlapping segments, computing periodograms, and averaging them.

**Parameters:**
- Window: 4-second Hamming windows (400 samples at 100 Hz)
- Overlap: 50% (200 samples)
- NFFT: 512 points (zero-padded for finer frequency resolution)

In [ ]:
BAND_NAMES = ['Delta', 'Theta', 'Alpha', 'Beta']
BAND_RANGES = {
    'Delta': (0.5, 4.0),
    'Theta': (4.0, 8.0),
    'Alpha': (8.0, 13.0),
    'Beta':  (13.0, 30.0),
}
BAND_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
STAGE_COLORS = ['#3366cc', '#dc3912', '#ff9900', '#109618', '#990099']

nperseg = int(4 * sfreq)
noverlap = nperseg // 2

stage_psds = {}
stage_freqs = None

for stage_idx, stage_name in enumerate(STAGE_NAMES):
    mask = y == stage_idx
    stage_epochs = X[mask]
    n_epochs = stage_epochs.shape[0]
    print(f"{stage_name:5s}: {n_epochs:4d} epochs", end='')

    all_psds = []
    for epoch in stage_epochs:
        freqs, psd = signal.welch(epoch, fs=sfreq, nperseg=nperseg,
                                  noverlap=noverlap, nfft=512)
        all_psds.append(psd)
        if stage_freqs is None:
            stage_freqs = freqs

    mean_psd = np.mean(all_psds, axis=0)
    stage_psds[stage_name] = mean_psd
    print(f"  |  mean PSD shape: {mean_psd.shape}")

print(f"\nFrequency resolution: {stage_freqs[1] - stage_freqs[0]:.3f} Hz")

---
## 3. Average Band Power Calculation

For each stage, we integrate (trapezoidal rule) the PSD within each classical frequency band. This gives a single number per band per stage representing the average power.

In [ ]:
def band_power(freqs, psd, band):
    lo, hi = band
    mask = (freqs >= lo) & (freqs <= hi)
    return np.trapz(psd[mask], freqs[mask])

band_powers = {}  # stage_name -> {band_name: power}

print(f"{'Stage':>6s}  {'Delta':>10s} {'Theta':>10s} {'Alpha':>10s} {'Beta':>10s}")
print("-" * 50)

for stage_idx, stage_name in enumerate(STAGE_NAMES):
    mask = y == stage_idx
    stage_epochs = X[mask]
    n_epochs = stage_epochs.shape[0]
    if n_epochs == 0:
        continue

    powers = {band: [] for band in BAND_NAMES}
    for epoch in stage_epochs:
        freqs, psd = signal.welch(epoch, fs=sfreq, nperseg=nperseg,
                                  noverlap=noverlap, nfft=512)
        for band_name in BAND_NAMES:
            powers[band_name].append(band_power(freqs, psd, BAND_RANGES[band_name]))

    means = {b: np.mean(powers[b]) for b in BAND_NAMES}
    band_powers[stage_name] = means

    print(f"{stage_name:>6s}  {means['Delta']:10.2e} {means['Theta']:10.2e} "
          f"{means['Alpha']:10.2e} {means['Beta']:10.2e}")

---
## 4. Visualization: PSD Curves

Overlaid mean PSD curves for all five stages. This is the core EEG physiology plot — it reveals how brain oscillations shift with sleep depth.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for stage_idx, stage_name in enumerate(STAGE_NAMES):
    if stage_name not in stage_psds:
        continue
    ax.semilogy(stage_freqs, stage_psds[stage_name],
                color=STAGE_COLORS[stage_idx], label=stage_name,
                linewidth=1.5)

for band_name, (lo, hi) in BAND_RANGES.items():
    ax.axvspan(lo, hi, alpha=0.06, color='gray')
    mid = (lo + hi) / 2
    ax.text(mid, ax.get_ylim()[1] * 0.98, band_name,
            ha='center', va='top', fontsize=9, alpha=0.5,
            fontstyle='italic')

ax.set_xlim(0, 32)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density (V²/Hz)')
ax.set_title('Mean Power Spectral Density by Sleep Stage')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('figures/psd_by_stage.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/psd_by_stage.png")

### PSD Curve Interpretation

**N3 (deep sleep)** shows markedly elevated power in the delta band (0.5–4 Hz), visible as a strong peak at low frequencies. This reflects the synchronized firing of cortical neurons producing high-amplitude slow-wave activity — the hallmark of deep sleep.

**Wake** has higher power in the alpha (8–13 Hz) and beta (13–30 Hz) ranges, associated with alert wakefulness and active information processing. The alpha rhythm (posterior dominant rhythm) is especially prominent when eyes are closed.

**N2** shows intermediate patterns with some delta activity but less pronounced than N3, plus the signature sleep spindles (11–16 Hz bursts) visible as subtle bumps in the alpha/low-beta region.

**REM** and **N1** have PSD profiles similar to Wake — mixed-frequency, low-amplitude activity — which is why single-channel EEG alone struggles to separate these stages reliably.

---
## 5. Visualization: Band Power Bar Charts

Grouped bar charts comparing absolute and relative band power across stages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(STAGE_NAMES))
width = 0.18

# --- Absolute band power ---
ax = axes[0]
for i, band_name in enumerate(BAND_NAMES):
    values = [band_powers[s][band_name] for s in STAGE_NAMES]
    ax.bar(x + i * width - 1.5 * width, values, width,
           label=band_name, color=BAND_COLORS[i], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(STAGE_NAMES)
ax.set_ylabel('Absolute Band Power (V²)')
ax.set_title('Absolute Band Power by Sleep Stage')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

# --- Relative band power ---
ax = axes[1]
for i, band_name in enumerate(BAND_NAMES):
    values = []
    for s in STAGE_NAMES:
        total = sum(band_powers[s].values())
        values.append(band_powers[s][band_name] / total * 100)
    ax.bar(x + i * width - 1.5 * width, values, width,
           label=band_name, color=BAND_COLORS[i], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(STAGE_NAMES)
ax.set_ylabel('Relative Band Power (%)')
ax.set_title('Relative Band Power Distribution by Sleep Stage')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('figures/band_power_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/band_power_bars.png")

### Band Power Interpretation

| Stage | Dominant Band | Physiological Significance |
|-------|--------------|---------------------------|
| **Wake** | Alpha, Beta | Eyes-closed alpha rhythm (~10 Hz); beta reflects active cortical processing |
| **N1** | Theta | Drowsiness: alpha fragments, theta emerges (4–7 Hz) — the "transition" band |
| **N2** | Delta + Spindles | Light sleep: K-complexes and sleep spindles (sigma, ~12–15 Hz) superimposed on background delta |
| **N3** | Delta | Deep sleep: massive synchronized slow-wave activity (0.5–4 Hz, >75 μV); the deepest, most restorative stage |
| **REM** | Mixed (Theta + Beta) | Paradoxical sleep: active brain in paralyzed body; desynchronized EEG resembles wake |

**Why N3 has strong delta:** During deep sleep, thalamocortical neurons oscillate in a synchronized slow (<1 Hz) rhythm, which entrains cortical networks into widespread, high-amplitude slow waves. This is the hallmark of N3 and is essential for memory consolidation and metabolic restoration.

**Why Wake shows more alpha/beta:** Alpha rhythm is generated by thalamocortical loops when the thalamus is in a "relay" mode (not bursting), allowing sensory information to flow to the cortex. Beta reflects ongoing cortical computation. Both dissipate as the thalamus switches to burst mode during sleep.

**Why N1 and REM are difficult to separate:** Both stages exhibit low-amplitude, mixed-frequency EEG with similar spectral profiles. N1 is a brief transitional stage (often <5 minutes) with theta activity resembling the tonic background of REM. Without EOG (eye movement) and EMG (muscle tone) channels, distinguishing N1 from REM using a single EEG channel is inherently ambiguous — even human scorers rely on polygraphic context.

---
## 6. Quantitative Summary Table

In [ ]:
print(f"{'Stage':>6s}  {'Epochs':>6s}  {'Delta':>10s}  {'Theta':>10s}  {'Alpha':>10s}  {'Beta':>10s}")
print("=" * 56)
for stage_name in STAGE_NAMES:
    mask = y == STAGE_NAMES.index(stage_name)
    n_epochs = np.sum(mask)
    p = band_powers[stage_name]
    total = sum(p.values())
    rel = {b: p[b] / total * 100 for b in BAND_NAMES}
    print(f"{stage_name:>6s}  {n_epochs:6d}  "
          f"{p['Delta']:8.2e} ({rel['Delta']:5.1f}%)  "
          f"{p['Theta']:8.2e} ({rel['Theta']:5.1f}%)  "
          f"{p['Alpha']:8.2e} ({rel['Alpha']:5.1f}%)  "
          f"{p['Beta']:8.2e} ({rel['Beta']:5.1f}%)")

---
## Key Takeaways

1. **Delta power is the strongest discriminators of N3** — a >2× increase over all other stages, consistent with sleep homeostasis.
2. **Alpha/beta ratio distinguishes Wake from sleep** — Wake shows beta dominance; all sleep stages show a relative shift toward slower frequencies.
3. **N1 and REM are spectrally similar** — both show mixed theta/beta with minimal delta, explaining why single-channel EEG classifiers typically struggle with these stages.
4. **Absolute vs. relative power matters** — N3's delta dominance is obvious in absolute terms; relative normalization reveals subtler shifts (e.g., theta proportion increases from Wake → N1).
5. **Limitations:** This analysis uses a single EEG channel (Fpz-Cz) from one subject. Clinical sleep staging uses multiple EEG channels + EOG + EMG for reliable scoring.